# 02 | Limpeza e EDA

Executa tratamento de dados e revisão dos principais resultados exploratórios.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
import json
import pandas as pd
from src.clean_data import execute as clean_execute
from src.create_features import execute as feature_execute
from src.exploratory_analysis import execute as eda_execute
from src.config import PROCESSED_CLIENTES, MODEL_METRICS_FILE, SUMMARY_FILE


In [ ]:
clean_execute()
feature_execute()
eda_execute()

## 3 | Análise Probabilística — Churn por Categoria de Comportamento

Requisito do hackathon: _"Uso de fundamentos de probabilidade para identificar produtos/perfis de maior destaque, justificando-os com análises estatísticas."_

Calculamos:
- **Probabilidade condicional** P(Churn | Perfil de Risco)
- **Probabilidade condicional** P(Churn | Estado)
- **Probabilidade condicional** P(Churn | Membro Ativo)
- **Teste Qui-Quadrado** para verificar independência estatística entre perfil de risco e churn
- **Risco Relativo** de cada segmento vs. baseline

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
from src.config import PROCESSED_CLIENTES, FEATURE_BASE

clientes = pd.read_csv(PROCESSED_CLIENTES)
base = pd.read_csv(FEATURE_BASE)

# ─────────────────────────────────────────────────────────────
# 1. Probabilidade baseline de churn
# ─────────────────────────────────────────────────────────────
p_churn_global = clientes['churn_flag'].mean()
print(f"📊 Probabilidade global de churn: {p_churn_global:.2%}")
print(f"   → De cada 100 clientes, aproximadamente {p_churn_global*100:.0f} cancelam.\n")

# ─────────────────────────────────────────────────────────────
# 2. P(Churn | Perfil de Risco) — probabilidade condicional
# ─────────────────────────────────────────────────────────────
prob_perfil = (
    clientes.groupby('perfil_risco')['churn_flag']
    .agg(['mean', 'count', 'sum'])
    .rename(columns={'mean': 'P(Churn|Perfil)', 'count': 'Total', 'sum': 'Churners'})
    .sort_values('P(Churn|Perfil)', ascending=False)
)
prob_perfil['Risco Relativo vs Baseline'] = prob_perfil['P(Churn|Perfil)'] / p_churn_global
prob_perfil['P(Churn|Perfil)'] = prob_perfil['P(Churn|Perfil)'].map('{:.2%}'.format)
prob_perfil['Risco Relativo vs Baseline'] = prob_perfil['Risco Relativo vs Baseline'].map('{:.2f}x'.format)
print("📌 P(Churn | Perfil de Risco):")
display(prob_perfil)

# ─────────────────────────────────────────────────────────────
# 3. P(Churn | Estado) — probabilidade condicional por região
# ─────────────────────────────────────────────────────────────
prob_estado = (
    clientes.groupby('estado')['churn_flag']
    .agg(['mean', 'count', 'sum'])
    .rename(columns={'mean': 'P(Churn|Estado)', 'count': 'Total', 'sum': 'Churners'})
    .sort_values('P(Churn|Estado)', ascending=False)
)
prob_estado['Risco Relativo vs Baseline'] = (prob_estado['P(Churn|Estado)'] / p_churn_global).map('{:.2f}x'.format)
prob_estado['P(Churn|Estado)'] = prob_estado['P(Churn|Estado)'].map('{:.2%}'.format)
print("\n📌 P(Churn | Estado):")
display(prob_estado)

# ─────────────────────────────────────────────────────────────
# 4. P(Churn | Membro Ativo)
# ─────────────────────────────────────────────────────────────
prob_membro = clientes.groupby('membro_ativo')['churn_flag'].mean()
p_churn_inativo = prob_membro.get(0, 0)
p_churn_ativo   = prob_membro.get(1, 0)
print(f"\n📌 P(Churn | Membro Inativo): {p_churn_inativo:.2%}")
print(f"   P(Churn | Membro Ativo):   {p_churn_ativo:.2%}")
print(f"   → Clientes inativos têm {p_churn_inativo/p_churn_ativo:.2f}x mais probabilidade de churn.")

# ─────────────────────────────────────────────────────────────
# 5. Teste Qui-Quadrado: independência entre Perfil e Churn
# ─────────────────────────────────────────────────────────────
tabela_contingencia = pd.crosstab(clientes['perfil_risco'], clientes['churn_flag'])
chi2, p_valor, graus_liberdade, esperados = stats.chi2_contingency(tabela_contingencia)
print(f"\n📐 Teste Qui-Quadrado — Perfil de Risco vs Churn:")
print(f"   χ² = {chi2:.4f} | Graus de liberdade = {graus_liberdade} | p-valor = {p_valor:.4f}")
if p_valor < 0.05:
    print(f"   ✅ p-valor < 0.05 → Rejeitamos H₀. Perfil de risco e churn são estatisticamente dependentes.")
    print(f"      O perfil do cliente é um preditor significativo de churn (não é resultado do acaso).")
else:
    print(f"   H₀ não rejeitada: não há evidência estatística de dependência (p={p_valor:.4f}).")

# ─────────────────────────────────────────────────────────────
# 6. Correlação de Pearson entre variáveis numéricas e churn
# ─────────────────────────────────────────────────────────────
cols_numericas = ['idade', 'renda_mensal', 'saldo_atual', 'score_credito', 'tempo_relacionamento', 'produtos_ativos']
correlacoes = clientes[cols_numericas + ['churn_flag']].corr()['churn_flag'].drop('churn_flag').sort_values(key=abs, ascending=False)
print("\n📐 Correlação de Pearson entre variáveis numéricas e churn_flag:")
for col, corr in correlacoes.items():
    _, p_corr = stats.pearsonr(clientes[col], clientes['churn_flag'])
    sig = '✅ significativo' if p_corr < 0.05 else '❌ não significativo'
    print(f"   {col:30s}: r = {corr:+.4f} | p-valor = {p_corr:.4f} | {sig}")

print("\n✅ Análise probabilística concluída.")

In [ ]:
display(pd.read_csv(PROCESSED_CLIENTES).describe(include='all').T.head(12))
json.loads(Path(SUMMARY_FILE).read_text(encoding='utf-8'))
